# 🎓 Programming in Python (M4010104 / M4020104)
# Lecture 1 & 2 Master Review: Architecture, Bytecode Compilation, Memory Model & Operators
**Instructor:** Dr. Sunil TT | **Department / Centre:** ICFOSS / DUK  
**Target Cohort:** M.Tech / M.Sc. Artificial Intelligence  

---

### 📖 Purpose of this Interactive Master Notebook:
This notebook serves as an in-depth instructor presentation and review tool. For each foundational topic, we systematically break down:
1. **Conceptual Motivation:** The computational, algorithmic, and AI engineering context.
2. **Underlying Mechanism:** How CPython, heap allocations, the bytecode compiler, and the Python Virtual Machine (PVM) operate under the hood.
3. **Common Pitfalls & Gotchas:** Subtle bugs, performance traps, and edge cases that trip up engineers and data scientists.
4. **Interactive Probes:** Code demonstrations illustrating both correct idioms and edge-case behavior.

---
## 1. Runtime Environment, Host System & Architecture Introspection

In AI/ML and scientific computing, pipelines depend heavily on hardware acceleration (CPUs, GPUs, TPUs) and C-extensions (NumPy, PyTorch, TensorRT). Knowing your exact runtime implementation (CPython vs. PyPy), pointer architecture (64-bit vs. 32-bit), interpreter version, and environment isolation is critical for reproducibility, binary package compatibility, and debugging ABI mismatches.

### 1.1 Python Specification vs. Implementations: CPython & PyPy
Many beginners do not realize that **Python** is a language specification (syntax, grammar, standard library rules), whereas the program executing your code is an **implementation**.

| Implementation | Core Engine | Key Mechanism | Best Use Case |
| :--- | :--- | :--- | :--- |
| **CPython** (Standard) | Written in **C** | Compiles `.py` to bytecode, executed by the C-based PVM. Uses Global Interpreter Lock (GIL). | Standard AI/ML, Data Science, NumPy/PyTorch ecosystem. |
| **PyPy** | Written in **RPython** | Features a **Just-In-Time (JIT) Compiler** that compiles hot loops into native machine code at runtime (4x–7x speedup). | Pure Python algorithms, web backends, financial simulations without heavy C-extensions. |
| **MicroPython / CircuitPython** | Written in **C** | Ultra-lean interpreter optimized for microcontrollers (ESP32, RP2040, STM32). | Embedded systems, robotics, IoT telemetry. |

> **💡 Why Data Science defaults to CPython:** Although PyPy is faster for pure Python arithmetic, data science libraries (NumPy, SciPy, Pandas, PyTorch) are already written in hand-optimized C/C++/CUDA. CPython interfaces with these native C-libraries with zero overhead via the C-API.

### 1.2 Python Virtual Environments: Why and How
When you install Python packages globally using `pip install`, they land in a single shared system directory (`site-packages`).

#### ⚠️ The "Dependency Collision" Problem:
* **Project A (Legacy Computer Vision):** Requires `opencv-python==3.4.2` and `numpy==1.19.5`.
* **Project B (Generative AI / LLM):** Requires `transformers==4.38.0` and `numpy==1.26.4`.
* Installing Project B globally will overwrite NumPy, instantly breaking Project A!

#### 🛡️ The Solution: Isolated Virtual Environments (`venv`)
A virtual environment is a lightweight, self-contained directory tree containing its own Python binary and isolated `site-packages` directory. It modifies your shell's `PATH` so `python` and `pip` point to the environment's private sandbox.

#### 🛠️ Essential Commands for Beginners:
```bash
# 1. Create a virtual environment named 'ai_env'
python3 -m venv ai_env

# 2. Activate the environment:
# On Linux / macOS:
source ai_env/bin/activate
# On Windows:
ai_env\Scripts\activate

# 3. Install packages safely inside the active sandbox
pip install numpy pandas matplotlib

# 4. Freeze exact dependency versions for team reproducibility
pip freeze > requirements.txt

# 5. Recreate environment on another machine
pip install -r requirements.txt

# 6. Register environment as a Jupyter Kernel
python -m ipykernel install --user --name=ai_env --display-name "Python (ai_env)"
```

### ⚙️ Underlying Mechanism:
* **CPython** is the reference implementation written in C. Python source code (`.py`) is compiled into intermediate bytecode (`.pyc`), executed by the Python Virtual Machine (PVM) evaluation loop.
* When a virtual environment is activated, Python reads `pyvenv.cfg` located in the environment directory, setting `sys.prefix` and isolating package lookup from the global system interpreter.
* The `sys` and `platform` modules query the host operating system and C-runtime bindings directly to reveal interpreter build parameters.

### ⚠️ Potential Pitfalls & Gotchas:
* **Virtual Environment Confusion:** Running a Jupyter notebook inside the wrong kernel where required packages are missing. Check `sys.executable` in the code cell below to verify your active environment path!
* **Architecture Mismatches:** On modern ARM architectures (e.g., Apple Silicon, Raspberry Pi, AWS Graviton), running an x86_64 translated binary under emulation leads to silent 3x–5x performance degradation.

In [ ]:
import sys
import platform
import os

# 1. Implementation & Compiler Diagnostics
print(f"Python Implementation : {platform.python_implementation()} (e.g., CPython / PyPy)")
print(f"Python Version        : {sys.version.split()[0]} ({platform.python_build()[0]})")
print(f"Compiler              : {platform.python_compiler()}")
print(f"System Architecture   : {platform.machine()} ({platform.architecture()[0]})")
print(f"OS Platform           : {sys.platform}")

# 2. Virtual Environment Detection
in_venv = (sys.prefix != sys.base_prefix)
print("\n--- Virtual Environment Diagnostics ---")
print(f"Active in Virtual Env : {in_venv}")
print(f"Active Python Path    : {sys.executable}")
print(f"Active sys.prefix     : {sys.prefix}")
print(f"Base System Prefix    : {sys.base_prefix}")

Python Implementation : CPython (e.g., CPython / PyPy)
Python Version        : 3.12.13 (main)
Compiler              : GCC 11.4.0
System Architecture   : x86_64 (64bit)
OS Platform           : linux

--- Virtual Environment Diagnostics ---
Active in Virtual Env : False
Active Python Path    : /usr/bin/python3
Active sys.prefix     : /usr
Base System Prefix    : /usr


---
## 2. CPython Bytecode Compilation & Virtual Machine Disassembly (`dis`)

Python is often described loosely as an "interpreted" language. In reality, CPython is a **bytecode-compiled, stack-based virtual machine**. Understanding how high-level Python code is translated into bytecode instructions demystifies:
1. Why certain Python idioms (like list comprehensions or built-in functions) run significantly faster than explicit loops.
2. How local variable lookup (`LOAD_FAST`) differs from global lookup (`LOAD_GLOBAL`).
3. How control flow constructs (branches, loops) are evaluated at the machine level.

### ⚙️ Underlying Mechanism:
The CPython execution pipeline follows 4 clear stages:
$$\text{Source Code (.py)} \xrightarrow{\text{Tokenizer/Parser}} \text{Abstract Syntax Tree (AST)} \xrightarrow{\text{Compiler}} \text{Code Object (Bytecode)} \xrightarrow{\text{PVM Loop}} \text{Machine Execution}$$

* **Code Objects (`PyCodeObject`):** A compiled immutable chunk of executable code containing:
  * `co_code`: The raw sequence of bytecode instruction opcodes and arguments.
  * `co_consts`: Tuple of literals/constants used in the code (e.g., numbers, strings).
  * `co_varnames`: Tuple of local variable names (accessed by fast integer array indices in C).
  * `co_names`: Tuple of global names and attribute names.
  * `co_stacksize`: The maximum virtual machine evaluation stack depth needed.
* **The CPython Stack Machine (`ceval.c`):** CPython executes bytecode on an internal evaluation stack. For example, adding two numbers `a + b` involves: `LOAD_FAST` (push `a`), `LOAD_FAST` (push `b`), `BINARY_OP` (pop both, add, push result), and `STORE_FAST` (pop result, store into target).
* **The `__pycache__` Directory:** When a module is imported, CPython caches the compiled bytecode in a `.pyc` file in `__pycache__/` to skip the parsing/compilation stage on subsequent runs.

### ⚠️ Potential Pitfalls & Gotchas:
* **Misconception: Bytecode is Native Machine Code.** Bytecode is NOT binary CPU machine code (like x86 or ARM assembly). It is an intermediate instruction set interpreted by the C-based loop in `Python/ceval.c`. It does not provide CPU-native execution speed unless JIT-compiled (e.g., in PyPy or Numba).
* **No Bytecode Cross-Version Portability:** Bytecode generated by Python 3.10 will NOT run on Python 3.11 or 3.12. Python 3.11+ introduced the *Specializing Adaptive Interpreter* (PEP 659) which dynamically modifies bytecode opcodes at runtime (e.g., `BINARY_OP_ADD_INT`) for optimization.
* **Bytecode Is Not Encrypted:** Never store private API keys or secrets in `.pyc` files assuming they are protected. Bytecode can be decompiled back to near-original Python source code in seconds using disassemblers.

In [ ]:
import dis

# Define two functions with equivalent logic to compare their bytecode instructions
def compute_loop(n):
    total = 0
    for i in range(n):
        total += i
    return total

def compute_builtin(n):
    return sum(range(n))

print("=" * 65)
print("1. DISASSEMBLY OF compute_loop(n):")
print("=" * 65)
dis.dis(compute_loop)

print("\n" + "=" * 65)
print("2. DISASSEMBLY OF compute_builtin(n):")
print("=" * 65)
dis.dis(compute_builtin)

print("\n" + "=" * 65)
print("3. INSPECTING CODE OBJECT ATTRIBUTES (compute_loop):")
print("=" * 65)
co = compute_loop.__code__
print(f"co_varnames (Local Variables) : {co.co_varnames}")
print(f"co_consts   (Constants/Literals): {co.co_consts}")
print(f"co_names    (Global Identifiers): {co.co_names}")
print(f"co_stacksize(Max Stack Depth)   : {co.co_stacksize}")
print(f"co_code     (Raw Byte Sequence) : {list(co.co_code[:12])}...")

1. DISASSEMBLY OF compute_loop(n):
  4           0 RESUME                   0

  5           2 LOAD_CONST               1 (0)
              4 STORE_FAST               1 (total)

  6           6 LOAD_GLOBAL              1 (NULL + range)
             16 LOAD_FAST                0 (n)
             18 CALL                     1
             26 GET_ITER
        >>   28 FOR_ITER                 7 (to 46)
             32 STORE_FAST               2 (i)

  7          34 LOAD_FAST                1 (total)
             36 LOAD_FAST                2 (i)
             38 BINARY_OP               13 (+=)
             42 STORE_FAST               1 (total)
             44 JUMP_BACKWARD            9 (to 28)

  6     >>   46 END_FOR

  8          48 LOAD_FAST                1 (total)
             50 RETURN_VALUE

2. DISASSEMBLY OF compute_builtin(n):
 10           0 RESUME                   0

 11           2 LOAD_GLOBAL              1 (NULL + sum)
             12 LOAD_GLOBAL              3 (NULL + range)

---
## 3. Interactive Profiling & Benchmarking (`%timeit`)

Writing clean Python code often involves choosing between multiple idioms (e.g., explicit `for` loops, list comprehensions, generator expressions, or vectorized operations). Profiling execution time accurately allows engineers to identify computational bottlenecks without guess-work.

### ⚙️ Underlying Mechanism:
* `%timeit` is an IPython magic command that executes code thousands of times in multiple runs to calculate statistically rigorous execution times.
* **Bytecode Overhead in Loops:** In a pure Python `for` loop, each iteration executes multiple PVM bytecode instructions (`FOR_ITER`, `STORE_FAST`, `LOAD_FAST`, `BINARY_OP`, `INPLACE_ADD`). In contrast, list comprehensions and C-based built-ins run their iteration loops directly in optimized C-code inside the interpreter binary.

### ⚠️ Potential Pitfalls & Gotchas:
* **Single-Run Skew (`%time` vs `%timeit`):** Using `%time` only measures a single run, which is heavily susceptible to background OS interrupts and garbage collection spikes. `%timeit` disables garbage collection during timing runs to measure pure algorithmic latency.
* **Side-Effect Trap:** Benchmarking expressions with destructive mutations (e.g., `%timeit my_list.pop()`) will fail on subsequent iterations unless state is re-initialized.

In [ ]:
# Demonstrating algorithmic overhead: Traditional loop vs. Generator comprehension
def sum_squares_loop(n):
    total = 0
    for i in range(n):
        total += i * i
    return total

def sum_squares_builtin(n):
    return sum(i * i for i in range(n))

N = 100_000

print(f"Benchmarking calculation for N = {N:,} iterations:")
print("1. Traditional Loop:")
%timeit sum_squares_loop(N)

print("\n2. Built-in sum() with Generator Comprehension:")
%timeit sum_squares_builtin(N)

---
## 4. Kernel State & Namespace Introspection (`%whos` & Memory)

Jupyter notebooks maintain a persistent interactive session. When prototyping large AI models or loading multi-gigabyte datasets, tracking what objects are currently occupying kernel RAM is vital to avoid Out-Of-Memory (OOM) crashes.

### ⚙️ Underlying Mechanism:
* The IPython kernel stores all active variables inside the global namespace dictionary `globals()`.
* `%whos` inspects this dictionary and queries `type()` and data dimensions/lengths for each variable.

### ⚠️ Potential Pitfalls & Gotchas:
* **The Out-of-Order Execution Pitfall:** Cells can be executed in any arbitrary order in Jupyter (e.g., cell [12] before cell [3]). A variable modified in cell [12] persists in memory even if you scroll up and re-run cell [4], leading to non-reproducible phantom bugs.
* **Remedy:** Always test notebooks using **Kernel $\rightarrow$ Restart & Run All Cells** before sharing or submitting.

In [ ]:
cohort_title = "M.Tech AI 2026"
batch_size = 64
learning_rate = 0.001
sample_weights = [0.15, 0.35, 0.50]

# Display all active variables in kernel namespace
%whos

---
## 5. Object Referencing Model: Value Equality (`==`) vs. Object Identity (`is`)

Understanding the difference between **value equality** (having the same content) and **identity** (referencing the exact same object in memory) is the single most important mental model in Python. Without it, unintended aliasing bugs will silently corrupt data.

### ⚙️ Underlying Mechanism:
* In Python, variables are **labels (pointers)** bound to heap-allocated `PyObject` structures. Each object contains metadata (type tag, reference count `ob_refcnt`) and actual value data.
* `a == b`: Calls `a.__eq__(b)` to compare whether values match.
* `a is b`: Compares actual memory addresses `id(a) == id(b)`.
* Assignment (`y = x`) does **NOT** copy the underlying object; it merely binds the new name `y` to the existing memory address of `x`.

### ⚠️ Potential Pitfalls & Gotchas:
* **The Accidental Mutation Trap:** Modifying a mutable collection through reference `y` unexpectedly alters `x`.
* **Misuse of `is`:** Never use `is` to check equality for numbers or strings (e.g., `if status is "active":`). `is` should strictly be reserved for singleton checks like `if var is None:` or `if flag is True:`.

In [ ]:
# Demonstrating Object Referencing & Aliasing
x = [1, 2, 3]
y = x              # y points to the EXACT same list object in memory
z = [1, 2, 3]       # z is a BRAND NEW list object allocated with identical content

print(f"x == y: {x == y:<5} | x is y: {x is y:<5} | id(x) = {id(x)}, id(y) = {id(y)}")
print(f"x == z: {x == z:<5} | x is z: {x is z:<5} | id(x) = {id(x)}, id(z) = {id(z)}")

# Mutating object via tag 'y'
y.append(99)
print("\n--- After y.append(99) ---")
print(f"x: {x}  <-- Notice: x is modified because x and y reference the same object!")
print(f"z: {z}  <-- z is completely untouched")

---
## 6. Integer Interning & Small Integer Caching

Programmers coming from C/C++ or Java often observe bewildering behavior where `a is b` evaluates to `True` for some numbers, but `False` for others. Knowing Python's internal memory caching prevents writing buggy conditional checks.

### ⚙️ Underlying Mechanism:
* CPython maintains a pre-allocated global array of singleton integer objects for values in the range **$[-5, 256]$**.
* Whenever an integer within this range is referenced, CPython returns the pointer to the pre-existing singleton object rather than allocating new heap memory.
* For numbers $\ge 257$ or $\le -6$, CPython allocates distinct `PyLongObject` instances when created in separate interactive scopes.

### ⚠️ Potential Pitfalls & Gotchas:
* **False Sense of Correctness:** A unit test using `a is 10` will pass during testing, but the production code will silently fail when traffic inputs exceed `256`!

In [ ]:
# Testing small integer caching boundary
p1 = 256
p2 = 256
print(f"p1 = 256, p2 = 256 -> p1 is p2 : {p1 is p2} (id(p1) == id(p2))")

q1 = 257
q2 = 257
print(f"q1 = 257, q2 = 257 -> q1 is q2 : {q1 is q2} (in separate allocations)")
print(f"q1 == q2 (Value Equality)     : {q1 == q2}")

---
## 7. Primitive Data Types & Precision Mechanics

AI, optimization algorithms, and scientific simulations rely on numerical stability. Misunderstanding float representation or integer overflow limits can lead to catastrophic numerical drift or training divergence.

### ⚙️ Underlying Mechanism:
1. **Arbitrary-Precision Integers (`int`):** Unlike C (which overflows at $2^{31}-1$ or $2^{63}-1$), Python 3 integers automatically expand in size dynamically to accommodate numbers of any magnitude, limited only by system RAM.
2. **IEEE-754 Floating-Point Numbers (`float`):** Python floats are double-precision 64-bit binary fractions (53 bits significand, 11 bits exponent, 1 sign bit). Common decimal fractions like $0.1$ and $0.2$ cannot be represented exactly in binary and produce repeating fractions.
3. **Complex Numbers (`complex`):** Built-in native support for complex numbers with `.real` and `.imag` components and `abs()` computing Euclidean magnitude.
4. **Booleans as `int` Subclasses:** `bool` inherits directly from `int`. `True` has integer value `1`, and `False` has integer value `0`.

### ⚠️ Potential Pitfalls & Gotchas:
* **Direct Float Comparison (`f1 == f2`):** Never check `0.1 + 0.2 == 0.3`. Always use `math.isclose(a, b)` or `np.isclose(a, b)` with a tolerance threshold $\epsilon$.
* **Implicit Boolean Arithmetic:** Inadvertently doing mathematical operations on booleans (`True + True = 2`) can hide logical errors in data-cleaning pipelines.

In [ ]:
import math

# 1. Unbounded Arbitrary-Precision Integers
huge_int = 2 ** 128
print(f"2^128 = {huge_int}")
print(f"Memory occupied: {huge_int.bit_length()} bits ({sys.getsizeof(huge_int)} bytes)")

# 2. Floating-Point Representation Gotcha
f_calc = 0.1 + 0.2
f_target = 0.3
print("\n--- Floating-Point Precision ---")
print(f"0.1 + 0.2 == 0.3        : {f_calc == f_target}  <-- Dangerous exact comparison!")
print(f"0.1 + 0.2 exact in RAM  : {f_calc:.25f}")
print(f"0.3       exact in RAM  : {f_target:.25f}")
print(f"math.isclose(0.1+0.2, 0.3): {math.isclose(f_calc, f_target)}  <-- The robust engineering approach")

# 3. Native Complex Numbers
z = 3 + 4j
print(f"\nComplex z = {z} | Real: {z.real} | Imag: {z.imag} | |z|: {abs(z)}")

# 4. Boolean Integer Subclassing
print(f"\nisinstance(True, int)   : {isinstance(True, int)}")
print(f"True + True + False     : {True + True + False}")

---
## 8. Division, Modulo & Negative Number Mechanics

Indexing cyclical arrays, batch partitioning, FFT windowing, and spatial grid hashing depend on integer floor division (`//`) and remainder (`%`). Engineers moving from C, C++, or Java frequently introduce off-by-one bugs when dealing with negative indices.

### ⚙️ Underlying Mechanism:
* **True Division (`/`):** Always returns a `float`, even if the operands divide evenly (e.g., `4 / 2` yields `2.0`).
* **Floor Division (`//`):** Performs mathematical floor $\lfloor a / b \rfloor$, rounding towards **negative infinity** (unlike C/C++ which truncates towards zero).
* **Modulo Invariant:** Python strictly maintains the mathematical identity:
  $$a = (a // b) \times b + (a \% b)$$

### ⚠️ Potential Pitfalls & Gotchas:
* In C/C++: `-7 / 3 = -2` and `-7 % 3 = -1`.
* In Python: `-7 // 3 = -3` and `-7 % 3 = +2`.
* Failing to account for this difference when porting C/C++ audio DSP or embedded code to Python leads to incorrect indexing.

In [ ]:
print("--- Division Operators ---")
print(f" 7 /  3 = {7 / 3:.4f} (True float division)")
print(f" 7 // 3 = {7 // 3}      (Floor division)")
print(f"-7 // 3 = {-7 // 3}     (Floor division rounds towards -infinity)")
print(f"-7 %  3 = {-7 % 3}      (Python modulo guarantees remainder in range [0, divisor))")

# Verification of Python Modulo Invariant
a, b = -7, 3
invariant_check = (a == (a // b) * b + (a % b))
print(f"Invariant a == (a // b) * b + (a % b): {invariant_check}")

---
## 9. Chained Comparisons & Evaluation Order

Validating ranges (e.g., checking if probability $0 \le p \le 1$ or sensor reading is within tolerance) is ubiquitous in data processing.

### ⚙️ Underlying Mechanism:
* In Python, `a < b < c` translates to `(a < b) and (b < c)`.
* Crucially, the middle operand `b` is **evaluated only once** (unlike writing `(a < b) and (b < c)` where `b` might be an expensive function call evaluated twice).

### ⚠️ Potential Pitfalls & Gotchas:
* In C/C++, `10 < x < 20` compiles to `(10 < x) < 20` $\rightarrow$ `(1 or 0) < 20` $\rightarrow$ always `True`! In Python, it mathematically validates both bounds correctly.
* Non-intuitive chaining: `10 < x > 5` means `(10 < x) and (x > 5)`. Avoid writing zig-zag comparisons as they impair code readability.

In [ ]:
val = 15

# Clean range check
print(f"10 < val <= 20 : {10 < val <= 20}")
print(f"0 <= val < 10  : {0 <= val < 10}")

# Efficiency check: intermediate expression evaluated once
def get_reading():
    print("   [Executing sensor probe reading...]")
    return 18.5

print("Evaluating 10.0 <= get_reading() <= 25.0:")
is_valid = 10.0 <= get_reading() <= 25.0
print(f"Result: {is_valid}")

---
## 10. Short-Circuit Evaluation & Object Truthiness

Short-circuit evaluation is essential for defensive coding (preventing crashes from null pointers or division by zero) and writing concise default value fallbacks.

### ⚙️ Underlying Mechanism:
* **Truthiness:** Every object in Python has an inherent boolean value. Falsy values are: `None`, `False`, numeric zeros (`0`, `0.0`, `0j`), empty sequences/collections (`""`, `[]`, `()`, `{}`, `set()`). Everything else is truthy.
* `A or B`: Evaluates `A`. If `A` is truthy, Python **immediately returns `A` without evaluating `B`**. If `A` is falsy, it returns `B`.
* `A and B`: Evaluates `A`. If `A` is falsy, Python **immediately returns `A` without evaluating `B`**. If `A` is truthy, it returns `B`.

### ⚠️ Potential Pitfalls & Gotchas:
* **The Zero Override Bug:** Writing `timeout = user_timeout or 30` will incorrectly set `timeout = 30` when the user intentionally provided `user_timeout = 0` (because `0` is falsy).
* **Best Practice for Defaults:** Use explicit null checks: `timeout = user_timeout if user_timeout is not None else 30`.

In [ ]:
# 1. Short-Circuit OR returning operand value
default_config = {"lr": 0.01, "batch_size": 32}
user_config = None
active_config = user_config or default_config
print(f"Active Config from (user_config or default_config): {active_config}")

# 2. Safe ZeroDivision Guard
denominator = 0
# Denominator check short-circuits before the division is ever evaluated
safe_quotient = (denominator != 0) and (100 / denominator)
print(f"Guarded calculation with zero denominator: {safe_quotient}")

# 3. The 0 Trap with 'or' Defaulting
user_iterations = 0
flawed_default = user_iterations or 50      # BUG: 0 is falsy, so 50 is picked!
correct_default = user_iterations if user_iterations is not None else 50
print(f"Flawed default for 0 iterations : {flawed_default} (Incorrect!)")
print(f"Correct default for 0 iterations: {correct_default} (Correct!)")

---
## 11. Formatted String Literals (f-Strings) & Engineering Output

Creating telemetry logs, tabular model training statistics, scientific reports, and aligned ASCII charts requires formatting numbers with precise column widths, scientific notation, and rounding.

### ⚙️ Underlying Mechanism:
* f-strings (PEP 498) are evaluated at runtime by specialized PVM opcodes (`FORMAT_VALUE`), making them significantly faster than `%` or `str.format()`.
* Format specification mini-language syntax: `{value:flags width.precision type}`.

### ⚠️ Potential Pitfalls & Gotchas:
* **Floating Truncation vs. Rounding:** Formatter `{val:.2f}` rounds to nearest even (banker's rounding in Python), not simple truncation.
* **Backslash in Expressions:** Prior to Python 3.12, backslashes (`\`) were not allowed inside expression blocks within f-strings.

In [ ]:
sensor_name = "IMU_GYRO_Z"
sample_idx = 1042
raw_reading = 0.00348719
accuracy_ratio = 0.98754
mem_address = 4095

print("=" * 68)
print(f"| {'FIELD':<20} | {'RAW DATA':<18} | {'FORMATTED OUTPUT':<22} |")
print("=" * 68)
print(f"| {'Sensor Name':<20} | {sensor_name:<18} | {sensor_name:>22} |")
print(f"| {'Sample Index':<20} | {sample_idx:<18} | {sample_idx:010d}             |")
print(f"| {'Reading (Scientific)':<20} | {raw_reading:<18} | {raw_reading:12.3e} rad/s      |")
print(f"| {'Accuracy':<20} | {accuracy_ratio:<18} | {accuracy_ratio * 100:6.2f} %             |")
print(f"| {'Memory Register':<20} | {mem_address:<18} | 0x{mem_address:08X}           |")
print("=" * 68)

# Inline Debugging Specifier (Python 3.8+)
alpha = 0.05
beta = 12.4
print(f"Debug Output: {alpha=}, {beta=}, {alpha * beta=:.4f}")

---
## 12. In-Class Review Challenges & Discussion Puzzles

### Puzzle 1: Shallow Copies and Nested Mutability Trap
**Question for Class:** When we make a shallow copy of a nested list using `list(orig)` or `orig.copy()`, what happens if we mutate an inner element vs. appending an outer element?

In [ ]:
matrix_original = [[1, 2], [3, 4]]
matrix_shallow = list(matrix_original)   # Creates a new outer list, but inner references are SHARED!

# Mutation 1: Modify inner sub-list
matrix_shallow[0][0] = 99

# Mutation 2: Append new sub-list to shallow copy
matrix_shallow.append([5, 6])

print(f"matrix_original : {matrix_original}  <-- Notice: inner list was modified!")
print(f"matrix_shallow  : {matrix_shallow}  <-- Outer list has [5, 6], original does not")

### Puzzle 2: Bitwise Logic (`&`, `|`, `^`) vs. Boolean Logic (`and`, `or`, `not`)
**Question for Class:** Why does `6 and 3` return `3`, while `6 & 3` returns `2`?

In [ ]:
x = 6   # Binary: 0110
y = 3   # Binary: 0011

print(f"x = {x} ({bin(x)}), y = {y} ({bin(y)})")
print(f"1. Logical  'x and y' : {x and y:<5} (Evaluates truthiness: x is truthy, so returns y)")
print(f"2. Bitwise  'x & y'   : {x & y:<5} (Binary: {bin(x & y):>6} -> 0110 & 0011 = 0010 = 2)")
print(f"3. Bitwise  'x | y'   : {x | y:<5} (Binary: {bin(x | y):>6} -> 0110 | 0011 = 0111 = 7)")
print(f"4. Bitwise  'x ^ y'   : {x ^ y:<5} (Binary: {bin(x ^ y):>6} -> 0110 ^ 0011 = 0101 = 5)")
print(f"5. Left Shift 'x << 2' : {x << 2:<5} (Binary: {bin(x << 2):>6} -> 011000 = 24 = x * 2^2)")

---
## 🎓 Summary of Key Takeaways for Lectures 1 & 2:
1. **CPython Bytecode Machine:** Python code compiles to intermediate bytecode interpreted by a stack machine (`Python/ceval.c`). The `dis` module lets us inspect the exact opcodes executed.
2. **Variables are Names/Pointers:** Memory allocation is decoupled from name binding. Use `==` for value equality and `is` only for identity/singleton checks.
3. **Numeric Safety:** Integers never overflow in Python (arbitrary precision), but floating-point arithmetic is subject to IEEE-754 binary precision constraints. Use `math.isclose()`.
4. **Floor Division & Modulo:** Python rounds towards negative infinity (`//`), preserving $a = (a // b) * b + (a \% b)$ across all signs.
5. **Short-Circuit Evaluation:** `and` / `or` return the decisive operand's value, not a bare boolean. Be cautious when using `or` for default parameters when `0` is a valid input.
6. **Interactive Hygiene:** Always use **Restart Kernel & Run All** to eliminate out-of-order execution artifacts.